# Dermalens Data Engineering on Google Cloud

This notebook demonstrates an end-to-end data engineering workflow using the Dermalens dataset and Google Cloud Platform (GCP).

### What this notebook does
1. Installs the required Python libraries.
2. Configures GCP authentication and project settings.
3. Creates a Google Cloud Storage (GCS) bucket.
4. Downloads the raw dermalens-datasets and stores it in GCS.
5. Combines and deduplicates the default splits, and derives the binary malignant/benign label.
6. Creates stratified train/dev/test datasets and 8-fold cross-validation splits, grouped by `lesion_id` to avoid leakage.
7. Stores the resulting shards back in GCS.

> **Before running:** Replace the placeholder GCP project and bucket settings. Make sure your account has permissions for GCS.

## Dermalens Data

### Source Data Overall Description
- [Dataset README](./Dataset_README.md) contains detailed information
- Hugging Face Dermalens data is stored as [Apache Parquet](https://motherduck.com/learn/why-choose-parquet-table-file-format/) an open-source columnar file format: values for each column sit together on disk, so a query that needs a single column can skip the other columns; so engines read only columns as query names 
- Hugging Face data is pre-split into training, validation and test 
- Data will be checked for duplicates where all non-image columns (other than the original split) are identical
- Default split data will be combined
- Customized split for [cross validation](https://www.geeksforgeeks.org/machine-learning/cross-validation-machine-learning/)
- Images linked to the same `lesion_id` will always stay together: in the same partition, and on the same side of every cross-validation fold


### Dermalens Dataset
Source columns stored as Apache Parquet, plus columns added by this notebook
| Key | Type | Description | Example |
|---|---|---|---|
| image | image | RGB dermoscopic lesion image | `PIL.Image` / array |
| image_id | string | Unique image identifier | `ISIC_0027419` |
| lesion_id | string | Lesion identifier; a lesion can have several images | `HAM_0000118` |
| dx | string | Original diagnosis label from the source HAM10000 taxonomy | `melanoma`, `basal_cell_carcinoma`, `actinic_keratoses` |
| dx_type | string | How the diagnosis was confirmed | `histo`, `follow_up`, `consensus`, `confocal` |
| age | float | Patient age in years (can be missing) | `52.0` |
| sex | string | Patient sex | `male`, `female` |
| localization | string | Body site / lesion location | `face`, `scalp`, `back` |
| split | string | *Added in Cell 5:* original Hugging Face split | `train`, `validation`, `test` |
| label | int | *Added in Cell 5:* binary target derived from `dx` | `1` = malignant, `0` = benign |
| row_id | int | *Added in Cell 7:* stable row identifier | `0` |
| partition | string | *Added in Cell 8:* custom modeling partition | `train`, `dev`, `test` |

Hugging Face [Dermalens Dataset](https://huggingface.co/datasets/dheraingoud/dermalens-datasets/viewer?row=31)
![Data Snippet](./images/data-screenshot.png)

Hugging Face [Parquet Files](https://huggingface.co/datasets/dheraingoud/dermalens-datasets/commit/52bb1787931b301d610829724dc78c15c2cab797)
![Parquet Files](./images/parquet-files.png)

### Skin-cancer Prediction as Binary Classification
This dataset is a binary skin-cancer classification dataset derived from HAM10000. The target is created from the `dx` label:
- `melanoma`, `basal_cell_carcinoma`, and `actinic_keratoses` (1) → malignant
- the remaining classes (0) → benign

| dx class | label |
|---|---:|
| `melanoma` | 1 |
| `basal_cell_carcinoma` | 1 |
| `actinic_keratoses` | 1 |
| `melanocytic_Nevi` | 0 |
| `benign_keratosis-like_lesions` | 0 |
| `dermatofibroma` | 0 |
| `vascular_lesions` | 0 |


#### Dermalens Default Dataset Split
Hugging Face data is pre-split
NOTE: For this project, the default split data will be combined and re-partitioned (Cells 5 and 7) to be used for cross-validation downstream

| Default Split | Images | % of Total |
|---|---:|---:|
| Train | 9,577 | 71.7% |
| Validation | 2,492 | 18.7% |
| Test | 1,285 | 9.6% |
| Total | 13,354 | 100.0% |

The overall malignant rate is about 19.5% (~2,609 of 13,354 images, before deduplication).


# Dermalens Data Configuration in Project
- PROJECT_ID = "dh-infra-20122699-skin"
- BUCKET_NAME = "dermalens-skin"
- LOCATION = "US" 
- [Architecture overview](#architecture-overview)

## Storage and versioning decisions

**Raw data: GCS bucket, `gs://dermalens-skin/datasets/<version>/raw/`.** The raw data is about 3.7 GB, mostly dermoscopic image bytes, and is only ever read in bulk (whole splits for training and preprocessing), never queried row by row. An object store is the right fit: it is cheap, scales without administration, is durable, and is directly readable by Colab/Vertex AI and by Pandas through `gcsfs`. The raw files are kept untouched so every later step can be regenerated from them.

**Processed data: GCS, `datasets/<version>/combined_data/` and `datasets/<version>/sharded/`, stored as Apache Parquet.** Parquet is columnar, compressed and typed. Metadata columns (`lesion_id`, `dx`, `age`, ...) can be read without loading the image bytes, which is what the splitting code does, and the schema (string, float, int, image bytes) survives round trips, unlike CSV. The partitions (`train`, `dev`, `test`) and the 8 CV folds are separate files, so a training job reads only the shard it needs.

**Object store, not a database.** There is no need for transactions, concurrent writers, joins or low-latency point lookups. The access pattern is write-once, read-whole-file, and the metadata is a handful of columns that fit comfortably in memory. A database would add cost and operations without benefit. If ad-hoc SQL over the metadata is needed later, the same parquet files can be loaded into BigQuery as an external table without changing the pipeline.

**Versioning.** 
- Each dataset version is identified by `DATASET_VERSION` (for example `v1`) and written under its own prefix, `datasets/<version>/`, so earlier versions are never overwritten.
- The upstream source is pinned to a Hugging Face commit (`HF_REVISION`), so a refresh of the source data is an explicit change, not a silent one.
- Cell 10 writes `datasets/<version>/manifest.json` with the source revision, preprocessing parameters, seed, row counts, class balance, library versions and a short change note.
- To publish an update (new source revision, different dedupe rule, label mapping, or split seed/logic): bump `DATASET_VERSION`, update the change note in Cell 10, and re-run the notebook from Cell 2.

## Architecture overview

The pipeline uses GCS as the raw, combined and sharded object store. Every artifact lives under a versioned prefix, `gs://dermalens-skin/datasets/<DATASET_VERSION>/`.

```parquet
Public Hugging Face parquet (train / validation / test), pinned to HF_REVISION
       |
       v
   Local download
       |
       v
GCS: datasets/<version>/raw/
       |
       v
Combine splits + deduplicate + derive label
       |
       v
GCS: datasets/<version>/combined_data/
       |
       v
Train / Dev / Test split (stratified on label, grouped by lesion_id)
       |
       v
8-fold stratified grouped CV (train only)
       |
       v
GCS: datasets/<version>/sharded/  +  manifest.json
```

The `raw/` prefix preserves the original source data, `combined_data/` holds the deduplicated, labelled dataset, and `sharded/` contains derived datasets suitable for model development and validation. See [Storage and versioning decisions](#storage-and-versioning-decisions) for the rationale.

## Cell 1 - Setup & Environment Installation

Install the libraries used throughout the notebook. `gcsfs` enables Pandas to read `gs://...` paths directly, while `google-cloud-storage` provides the native GCS API.

In [ ]:
# Install required libraries
!pip install --upgrade google-cloud-storage pandas pyarrow scikit-learn gcsfs

## Cell 2 - Configuration & GCP Authentication

Set the project, bucket, source dataset and dataset version that the pipeline will use. Authentication depends on where the notebook runs.

### Data access
- **Source:** the public Hugging Face dataset `dheraingoud/dermalens-datasets`, pinned to a fixed commit (`HF_REVISION`) and loaded with `datasets.load_dataset`. No credentials are needed.
- **Storage:** all derived data is written to and read from the GCS bucket `gs://dermalens-skin` under the versioned prefix `datasets/<DATASET_VERSION>/`. Writes use the `google-cloud-storage` client (`upload_to_gcs`); reads use `pandas.read_parquet("gs://...")` through `gcsfs` (see the read-back check in Cell 10).
- **Credentials:** 
  - **Google Colab / Vertex AI:** interactive Google authentication (`auth.authenticate_user()`).
  - **Local machine / CI:** Application Default Credentials (`gcloud auth application-default login`).
- **Access control:** the bucket uses uniform bucket-level access, so permissions are IAM-only. Writing needs `roles/storage.objectAdmin` on the bucket (or `objectCreator` plus `objectViewer`); reading only needs `roles/storage.objectViewer`. Creating the bucket needs `roles/storage.admin` on the project.

> **Security note:** Never commit a service-account JSON key or any password to source control. Prefer Secret Manager, environment variables, or the notebook environment's secret store.

In [ ]:
import pandas as pd
from google.cloud import storage

# --- GCP Configuration ---
PROJECT_ID = "dh-infra-20122699-skin"
BUCKET_NAME = "dermalens-skin"
LOCATION = "US"

# --- Source data and dataset version ---
HF_DATASET = "dheraingoud/dermalens-datasets"
# Pinned Hugging Face commit so the download is reproducible (see "Parquet Files" link above)
HF_REVISION = "52bb1787931b301d610829724dc78c15c2cab797"

# Bump DATASET_VERSION whenever the source revision, dedupe rule, label mapping,
# or split logic/seed changes. Every GCS artifact lives under this prefix.
DATASET_VERSION = "v1"
GCS_PREFIX = f"datasets/{DATASET_VERSION}"

# Authentication: Google Colab / Vertex AI
# (locally, use Application Default Credentials: `gcloud auth application-default login`)
from google.colab import auth
auth.authenticate_user()

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}/{GCS_PREFIX}/")
print(f"Source: {HF_DATASET} @ {HF_REVISION[:8]}")

## Cell 3 - Create the GCS Bucket

Create a Google Cloud Storage Bucket in specified project if it does not already exist.

In [3]:
def create_gcs_bucket(bucket_name, project_id, location="US"):
    """Create a GCS bucket if it does not already exist."""
    client = storage.Client(project=project_id)

    if client.lookup_bucket(bucket_name):
        print(f"Bucket 'gs://{bucket_name}' already exists.")
        return

    bucket = client.bucket(bucket_name)
    bucket.iam_configuration.uniform_bucket_level_access_enabled = True
    bucket = client.create_bucket(bucket, location=location)
    print(f"Successfully created bucket: gs://{bucket.name}")


create_gcs_bucket(BUCKET_NAME, PROJECT_ID, LOCATION)

Successfully created bucket: gs://dermalens-skin


# Cell 4 - Download Raw Dermalens Data from Hugging Face and Upload to Google Bucket
The raw dataset is downloaded from Hugging Face at the pinned commit `HF_REVISION` and uploaded to the Google bucket (`datasets/<version>/raw/`) without modification. Keeping an untouched raw copy is useful for reproducibility and allows downstream transformations to be regenerated.

In [ ]:
!pip install -q datasets

from google.cloud import storage
from pathlib import Path
from datasets import load_dataset

out_dir = Path("data")
out_dir.mkdir(exist_ok=True)

print(f"Loading {HF_DATASET} @ {HF_REVISION[:8]} from Hugging Face...")
# Load the pinned revision of the dataset
dataset = load_dataset(HF_DATASET, revision=HF_REVISION)

# Iterate over each split and save it locally as a Parquet file
for split_name in dataset.keys():
    local_path = out_dir / f"{split_name}.parquet"
    print(f"Saving {split_name} split to {local_path.resolve()}...")
    dataset[split_name].to_parquet(local_path)

print("All splits successfully downloaded and saved!")
print(f"Dermalens data is saved locally to: {out_dir.resolve()}")

def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    """Upload a local file to a GCS object path."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(
        f"Uploaded {source_file_name} -> "
        f"gs://{bucket_name}/{destination_blob_name}"
    )

# Upload each split to GCS
for split_name in dataset.keys():
    local_path = out_dir / f"{split_name}.parquet"
    upload_to_gcs(BUCKET_NAME, str(local_path.resolve()), f"{GCS_PREFIX}/raw/{split_name}.parquet")


## Cell 5 - Combined raw default split datasets for custom data splits
Default train, validation and test data from Hugging Face is combined and cleaned

Data where all non-image columns including image_id and lesion_id are identical is considered duplicate and removed from the dataset

Expected de-duplication:
- Combined raw rows before dedupe: 13354
- Rows removed as duplicates: 3339
- Combined rows after dedupe: 10015
- Malignant rate: 0.195

In [ ]:
from pathlib import Path
import pandas as pd
from google.cloud import storage

# Diagnosis classes treated as malignant (label = 1); everything else is benign (label = 0)
MALIGNANT_CLASSES = {"melanoma", "basal_cell_carcinoma", "actinic_keratoses"}

# 1) Locate the previously downloaded raw split files
data_dir = Path("data")
combined_dir = Path("combined_data")
combined_dir.mkdir(exist_ok=True)

split_names = ["train", "validation", "test"]
split_files = {}

for split in split_names:
    matches = sorted(data_dir.glob(f"*{split}*.parquet"))
    if matches:
        split_files[split] = matches[0]
        print(f"Found {split}: {matches[0]}")
    else:
        print(f"Warning: no parquet file found for split '{split}' in {data_dir}")

if not split_files:
    raise FileNotFoundError(
        f"No dataset parquet files found in {data_dir}. "
        "Download the train/validation/test splits first."
    )

# 2) Read and combine all splits (train first, so it wins when duplicates are dropped)
frames = []
for split, file_path in split_files.items():
    frame = pd.read_parquet(file_path)
    frame["split"] = split  # original Hugging Face split
    frames.append(frame)

combined_df = pd.concat(frames, ignore_index=True)
print(f"Combined raw rows before dedupe: {len(combined_df)}")

# 3) Remove duplicates: rows identical in every column except the image bytes and the
# original split label (so a record present in two splits is also caught)
before_dedupe = len(combined_df)
dedupe_subset = [col for col in combined_df.columns if col not in ("image", "split")]
combined_df = combined_df.drop_duplicates(subset=dedupe_subset).reset_index(drop=True)
after_dedupe = len(combined_df)
print(f"Rows removed as duplicates: {before_dedupe - after_dedupe}")
print(f"Combined rows after dedupe: {after_dedupe}")

# 4) Derive the binary target from the diagnosis
assert combined_df["dx"].notna().all(), "Missing dx values; cannot derive label"
combined_df["label"] = combined_df["dx"].isin(MALIGNANT_CLASSES).astype(int)
print(f"Malignant rate: {combined_df['label'].mean():.3f}")

# 5) Save locally
local_combined_path = combined_dir / "dermalens_combined.parquet"
combined_df.to_parquet(local_combined_path, index=False)
print(f"Saved combined data locally to: {local_combined_path.resolve()}")

# 6) Upload to the versioned 'combined_data' folder in the bucket
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
destination_blob = f"{GCS_PREFIX}/combined_data/dermalens_combined.parquet"
blob = bucket.blob(destination_blob)
blob.upload_from_filename(str(local_combined_path))
print(f"Uploaded to: gs://{BUCKET_NAME}/{destination_blob}")

## Cell 6 — Skin Data Features & Preprocessing Pipeline

This cell defines every feature the model sees, where it comes from, and how it is transformed. The image carries the strongest signal; the structured covariates are secondary but still useful.

| Feature | Type | Source | Definition | Transformation |
| --- | --- | --- | --- | --- |
| **image** | image (bytes in parquet) | source `image` | dermoscopic lesion image | stored as-is in this pipeline; resizing and normalization are applied at model-input time, not here |
| **age** | float | source `age` | patient age in years | median imputation, 0–1 min-max scaling |
| **age_missing** | float (0/1) | constructed | 1 if `age` was missing | flag computed before imputation |
| **sex** | categorical | source `sex` | patient sex | missing → `unknown`, then one-hot encoded |
| **localization** | categorical | source `localization` | body site (15 sites); face and scalp are strongly linked to actinic keratoses and basal cell carcinoma | missing → `unknown`, then one-hot encoded |

**Target:** `label` (int, 1 = malignant, 0 = benign), constructed in Cell 5 from `dx`.

**Columns that are NOT model inputs:**
- `dx`: the source of the target; using it as a feature would leak the answer.
- `dx_type`: describes how the diagnosis was confirmed, which would not be known for a new patient.
- `lesion_id`: used only to group images when splitting (Cell 7).
- `image_id`, `row_id`: identifiers used for traceability.
- `split`, `partition`: bookkeeping columns for the original and the custom split.

**No leakage:** the pipeline below is only *defined* here. It is fit on the training part of each fold (see Cell 9), so the median age, the scaling range and the category lists never see validation, dev or test data.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder

NUMERIC_COL = "age"
CATEGORICAL_COLS = ["sex", "localization"]
FEATURE_COLUMNS = [NUMERIC_COL] + CATEGORICAL_COLS  # the image is handled separately
UNKNOWN_CATEGORY = "unknown"

# Recorded in the dataset manifest (Cell 10)
FEATURE_CONFIG = {
    "numeric": {NUMERIC_COL: {"impute": "median", "missing_flag": "age_missing", "scale": "minmax_0_1"}},
    "categorical": {c: {"impute": UNKNOWN_CATEGORY, "encode": "one_hot"} for c in CATEGORICAL_COLS},
    "image": "stored as-is; resize/normalize at model-input time",
    "target": "label",
    "not_model_inputs": ["dx", "dx_type", "lesion_id", "image_id", "row_id", "split", "partition"],
}

def _missing_flag(X):
    """1.0 where the value is missing, else 0.0."""
    return pd.DataFrame(X).isna().to_numpy().astype(float)

def _fill_unknown(X):
    """Replace missing categories (None or NaN) with a single 'unknown' category."""
    return pd.DataFrame(X).fillna(UNKNOWN_CATEGORY).astype(str).to_numpy()

def build_feature_pipeline():
    """Unfitted feature pipeline; fit it on a training fold only."""
    age = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", MinMaxScaler()),
    ])
    categorical = Pipeline([
        ("fill", FunctionTransformer(_fill_unknown, feature_names_out="one-to-one")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("age", age, [NUMERIC_COL]),
        ("age_missing", FunctionTransformer(
            _missing_flag, feature_names_out=lambda transformer, names: ["age_missing"]
        ), [NUMERIC_COL]),
        ("categorical", categorical, CATEGORICAL_COLS),
    ])

print("Model-input columns:", FEATURE_COLUMNS)

## Cell 7 — Partition Data: Train / Dev / Test + 8-Fold CV

We create a grouped, stratified split of the combined Dermalens data into train, dev, and test partitions using a 10-fold grouped split. The binary `label` is the stratification target, so the malignant rate stays similar across partitions, and `lesion_id` is the grouping key, so all images of the same lesion stay together. HAM10000-derived data contains multiple images per lesion, and a naive random split would leak near-duplicate lesion views into both training and evaluation data.

The first two held-out folds become test and dev, and the remaining rows form the training set, giving approximately 80/10/10 proportions. The 8-fold cross-validation split is then generated only from the training set (also stratified and grouped), leaving the dev and test partitions untouched for final evaluation.

Assertions at the end confirm that no `lesion_id` appears in more than one partition or on both sides of a cross-validation fold.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
TARGET = "label"
GROUP_COL = "lesion_id"
N_OUTER_FOLDS = 10  # 10 folds -> two held-out folds = 10% dev + 10% test, 80% train
N_CV_FOLDS = 8      # cross-validation folds inside the training set

# Load the combined Dermalens parquet from Cell 5, skipping the nested image column
# (images can be re-joined later via image_id / row_id)
combined_path = Path("combined_data") / "dermalens_combined.parquet"
all_cols = pd.read_parquet(combined_path).columns.tolist()
for required in (TARGET, GROUP_COL):
    assert required in all_cols, f"'{required}' missing from combined data; re-run Cell 5"

df = pd.read_parquet(combined_path, columns=[c for c in all_cols if c != "image"])
df.insert(0, "row_id", range(len(df)))
assert df[GROUP_COL].notna().all(), f"{GROUP_COL} has missing values"

print("Dataset shape:", df.shape)
display(df.head())

def grouped_folds(frame, n_splits):
    """Return the held-out row positions of each stratified, lesion-grouped fold."""
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    return [held for _, held in splitter.split(frame, frame[TARGET], groups=frame[GROUP_COL])]

# Train (80%) / Dev (10%) / Test (10%): the first two held-out folds of a 10-fold split
folds = grouped_folds(df, n_splits=N_OUTER_FOLDS)
test_df = df.iloc[folds[0]].copy()
dev_df = df.iloc[folds[1]].copy()
train_df = df.drop(index=df.index[list(folds[0]) + list(folds[1])]).copy()

# 8-fold cross-validation from TRAIN only
train_reset = train_df.reset_index(drop=True)
cv_folds = {}
for fold_idx, val_idx in enumerate(grouped_folds(train_reset, n_splits=N_CV_FOLDS)):
    val_mask = train_reset.index.isin(val_idx)
    cv_folds[fold_idx] = (
        train_reset[~val_mask].copy(),
        train_reset[val_mask].copy(),
    )

print(
    f"Splits complete: Train ({len(train_df)}), "
    f"Dev ({len(dev_df)}), Test ({len(test_df)}), CV Folds ({len(cv_folds)})"
)

print("\nMalignant (positive) rates:")
rates = {"train": train_df[TARGET].mean(), "dev": dev_df[TARGET].mean(), "test": test_df[TARGET].mean()}
rates.update({f"cv_fold_{i}_val": val[TARGET].mean() for i, (_, val) in cv_folds.items()})
print(pd.Series(rates, name="positive_rate").to_frame())

# Leakage checks: no lesion may appear in more than one partition or on both sides of a CV fold
def groups(frame):
    return set(frame[GROUP_COL])

assert not (groups(train_df) & groups(dev_df)), "lesion leakage train/dev"
assert not (groups(train_df) & groups(test_df)), "lesion leakage train/test"
assert not (groups(dev_df) & groups(test_df)), "lesion leakage dev/test"
for i, (fold_train, fold_val) in cv_folds.items():
    assert not (groups(fold_train) & groups(fold_val)), f"lesion leakage in CV fold {i}"

print(f"\nNo {GROUP_COL} overlap across train/dev/test or CV folds.")

# Cell 8 - Data organization step
This a data organization step, not a model-training step.
- preserve original split metadata
- create a separate modeling partition label
- reassemble and export the data in a structured, shardable format

Add the partition info to the data and save the splits and CV folds locally and to GCS under `datasets/<version>/sharded/`.
- It keeps the original Hugging Face split in split
- it adds a new column partition to each subset
- it merges train, dev, and test back into one DataFrame
- it sorts by row_id for stable ordering
- it prints the combined shape and previews the first rows
- it then writes the partitioned parquet files (and CV folds) locally and uploads them to the versioned `sharded/` prefix

In [ ]:
from pathlib import Path

# Note: the combined data already has a 'split' column holding the ORIGINAL Hugging Face
# split, so the new partition goes in its own 'partition' column.
train_df["partition"] = "train"
dev_df["partition"] = "dev"
test_df["partition"] = "test"

df_with_splits = pd.concat([train_df, dev_df, test_df]).sort_values("row_id")

print("DataFrame with 'partition' column created. Shape:", df_with_splits.shape)
display(df_with_splits.head())

sharded_dir = Path("sharded")
(sharded_dir / "cv_folds").mkdir(parents=True, exist_ok=True)

# Save all partitions in one file, plus one file per partition
outputs = {
    "dermalens_splits.parquet": df_with_splits,
    "train.parquet": train_df,
    "dev.parquet": dev_df,
    "test.parquet": test_df,
}
for fold_idx, (fold_train, fold_val) in cv_folds.items():
    outputs[f"cv_folds/fold_{fold_idx}_train.parquet"] = fold_train
    outputs[f"cv_folds/fold_{fold_idx}_val.parquet"] = fold_val

for name, frame in outputs.items():
    local_path = sharded_dir / name
    frame.to_parquet(local_path, index=False)
    upload_to_gcs(BUCKET_NAME, str(local_path), f"{GCS_PREFIX}/sharded/{name}")

# Cell 9 - Feature pipeline check (fit inside the fold)
The feature pipeline from Cell 6 is fit on the training part of cross-validation fold 0 and then applied to that fold's validation part. This is how it should be used for every fold, and for the final model (fit on the full training set, apply to dev and test), so that no statistic (median age, scaling range, category list) is learned from held-out data.

In [ ]:
# Demonstrate the feature pipeline on CV fold 0: fit on the fold's training part ONLY,
# then apply the fitted pipeline to the validation part.
fold_train, fold_val = cv_folds[0]

feature_pipeline = build_feature_pipeline()
X_train = feature_pipeline.fit_transform(fold_train[FEATURE_COLUMNS])
X_val = feature_pipeline.transform(fold_val[FEATURE_COLUMNS])
feature_names = feature_pipeline.get_feature_names_out()

assert X_train.shape[1] == X_val.shape[1] == len(feature_names)
assert not pd.isna(X_train).any() and not pd.isna(X_val).any(), "NaNs left after preprocessing"

print(f"Fold 0 train features: {X_train.shape}, validation features: {X_val.shape}")
print(f"{len(feature_names)} features:", list(feature_names))
print(f"Train age range after scaling: {X_train[:, 0].min():.2f} - {X_train[:, 0].max():.2f}")

# Cell 10 - Dataset manifest and read-back check
This cell makes the dataset version traceable and verifies data access:
- writes `manifest.json` (source revision, preprocessing parameters, seed, split method, row counts, class balance, library versions and a change note) next to the data in `datasets/<version>/`
- reads shards straight back from GCS with `pd.read_parquet("gs://...")` and checks the row counts against the in-memory partitions

Together with the pinned Hugging Face revision, the manifest is enough to rebuild the same dataset from the raw source.

In [ ]:
import json
import datetime
import importlib.metadata as importlib_metadata

# Short description of what changed in this dataset version; update it whenever DATASET_VERSION is bumped
DATASET_CHANGES = "Initial release: combined HF splits, deduplicated, binary label from dx, lesion-grouped 80/10/10 split and 8-fold CV."

def partition_summary(frame):
    return {"rows": int(len(frame)), "lesions": int(frame[GROUP_COL].nunique()),
            "malignant_rate": round(float(frame[TARGET].mean()), 4)}

manifest = {
    "dataset_version": DATASET_VERSION,
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "changes": DATASET_CHANGES,
    "source": {
        "hf_dataset": HF_DATASET,
        "hf_revision": HF_REVISION,
        "rows_before_dedupe": int(before_dedupe),
        "rows_after_dedupe": int(after_dedupe),
        "rows_after_dedupe_by_original_split": {
            k: int(v) for k, v in df_with_splits["split"].value_counts().items()
        },
    },
    "preprocessing": {
        "dedupe_subset": dedupe_subset,
        "malignant_classes": sorted(MALIGNANT_CLASSES),
        "features": FEATURE_CONFIG,
    },
    "split": {
        "seed": SEED,
        "stratify_on": TARGET,
        "group_by": GROUP_COL,
        "method": "StratifiedGroupKFold",
        "outer_folds": N_OUTER_FOLDS,
        "partitions": "fold 0 -> test, fold 1 -> dev, remaining folds -> train",
        "cv_folds_on_train": N_CV_FOLDS,
    },
    "partitions": {
        "train": partition_summary(train_df),
        "dev": partition_summary(dev_df),
        "test": partition_summary(test_df),
    },
    "libraries": {
        name: importlib_metadata.version(name)
        for name in ("pandas", "pyarrow", "scikit-learn", "datasets", "google-cloud-storage")
    },
    "gcs_prefix": f"gs://{BUCKET_NAME}/{GCS_PREFIX}/",
}

manifest_path = sharded_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
upload_to_gcs(BUCKET_NAME, str(manifest_path), f"{GCS_PREFIX}/manifest.json")
print(json.dumps(manifest, indent=2))

# Read-back check: load the shards straight from GCS (needs gcsfs) and compare with memory
train_check = pd.read_parquet(f"gs://{BUCKET_NAME}/{GCS_PREFIX}/sharded/train.parquet")
fold0_val_check = pd.read_parquet(f"gs://{BUCKET_NAME}/{GCS_PREFIX}/sharded/cv_folds/fold_0_val.parquet")
assert len(train_check) == len(train_df), "train shard row count mismatch"
assert len(fold0_val_check) == len(cv_folds[0][1]), "fold 0 val shard row count mismatch"
print(f"\nRead back from GCS: train={len(train_check)} rows, fold_0_val={len(fold0_val_check)} rows")

## Final checks and production considerations

At this point you should have, under `gs://<BUCKET_NAME>/datasets/<DATASET_VERSION>/`:

- The original Dermalens parquet splits (pinned Hugging Face revision) preserved under `raw/`.
- A combined, deduplicated, labelled dataset under `combined_data/`.
- Independent train, dev, and test datasets, grouped by `lesion_id`.
- Eight stratified cross-validation folds derived only from the training data.
- A `manifest.json` describing the version, parameters, counts and library versions.

### Further steps

1. **Keep versions immutable.** Never overwrite an existing `datasets/<version>/`; bump `DATASET_VERSION` instead.
2. **Use stable identifiers.** `image_id` is the source identifier and `row_id` is assigned in Cell 7 to trace records across partitions.
3. **Validate schemas.** Define an explicit schema rather than relying on inference when loading into a warehouse.
4. **Avoid embedded credentials.** Use IAM, Application Default Credentials, Secret Manager, or environment variables.
5. **Add data-quality checks.** Validate row counts, null rates, duplicate IDs, and class distributions after each transformation.
6. **Fit preprocessing inside each fold.** Use `build_feature_pipeline()` (Cell 6) fit on the fold's training part only, as shown in Cell 9.
7. **Handle images at model time.** Resizing and normalization are not part of this data pipeline; record their parameters with the model.

### Expected GCS structure

```text
gs://<BUCKET_NAME>/datasets/<DATASET_VERSION>/
├── manifest.json
├── raw/
│   ├── train.parquet
│   ├── validation.parquet
│   └── test.parquet
├── combined_data/
│   └── dermalens_combined.parquet
└── sharded/
    ├── dermalens_splits.parquet
    ├── train.parquet
    ├── dev.parquet
    ├── test.parquet
    └── cv_folds/
        ├── fold_0_train.parquet
        ├── fold_0_val.parquet
        ├── fold_1_train.parquet
        ├── fold_1_val.parquet
        └── ...
```